In [2]:
import pandas as pd
import re
import json
import ast
from os import fdopen, remove
from tempfile import mkstemp
from shutil import move, copymode
from pprint import pprint 
from io import StringIO
import os
import string
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def flatten(xss):
    return [x for xs in xss for x in xs]

### Daten veranschaulichen:
- Häufigkeit der KI Terme
- Häufigste Wörter
- TF-IDF Wörter Veranschaulichen
- length of sentences / number of clauses
- number / type of verbs
- number / type of adjektives

In [3]:
plots_dir = "plots/"

In [4]:
docs_ki = pd.read_json(StringIO(open('generated_data/KI_sentences_tfidf.json').read()), orient="index")

ki_docs = docs_ki[docs_ki['sentences'].isna() == False].reset_index()
non_ki_docs = docs_ki[docs_ki['sentences'].isna() == True].reset_index()

print(f"Number of KI docs: {len(ki_docs)}")
print(f"Number of other docs: {len(non_ki_docs)}")
non_ki_docs.head()

Number of KI docs: 0
Number of other docs: 152225


,index,sentences,top_words
0,6036703.txt,NaN,"['unterstützen', 'zwangshaft söder', 'genüssli..."
1,0224025.txt,NaN,"['zivilisten dramatischen', 'nkunda erschütter..."
2,5903496.txt,NaN,"['kommunist laufe', 'radikalisieren', 'kommuni..."
3,1184664.txt,NaN,"['andré', 'andré kunz', 'kunz', 'ishikawa prof..."
4,806049.txt,NaN,"['patienten fördern', 'patienten', 'gesundheit..."


In [5]:
docs_ai = pd.read_json(StringIO(open('generated_data/AI_sentences_tfidf.json').read()), orient="index")

ai_docs = docs_ai[docs_ai['sentences'].isna() == False].reset_index()
non_ai_docs = docs_ai[docs_ai['sentences'].isna() == True].reset_index()

print(f"Number of AI docs: {len(ai_docs)}")
print(f"Number of other docs: {len(non_ai_docs)}")
docs_ai.head()

Number of AI docs: 0
Number of other docs: 574


,sentences,top_words
5315432.txt,NaN,"['scooter 25', 'scooter', 'pushes scooter', 'p..."
5124643.txt,NaN,"['unarmed civilians', 'government forces', 'ba..."
5094180.txt,NaN,"['weiwei does', 'weiwei', 'shortcoming', 'nomi..."
851887.txt,NaN,"['philipp palm', 'philipp', 'palm crisis', 'pa..."
5455271.txt,NaN,"['world 2000', 'world', 'ki', '2000 ki', '2000']"


In [11]:
docs_sents = pd.read_csv(StringIO(open('generated_data/KI_sentences_translated.csv').read()))
docs_full = pd.read_csv(StringIO(open('generated_data/KI_sentences_full_translated.csv').read()))
docs_leipzig = pd.read_json(StringIO(open('generated_data/Leipzig_KI_sentences.json').read()), orient="index")

print(f"Number of leipzig 2024 docs: {len(docs_leipzig)}")
docs_full.head()

Number of leipzig 2024 docs: 3357


,id,text,translation
0,1739352_full.txt,Mit einer Mahnwache gegen die Todesstrafe und ...,With a vigil against the death penalty and a s...
1,1364384_full.txt,Berlin taz – Ein edleres Spendensiegel ist kau...,Berlin taz - A nobler seal of approval for don...
2,483913_full.txt,930 Friedrichstadtpalast Eu cand vreau sa flui...,930 Friedrichstadtpalast Eu cand vreau sa flui...
3,1305430_full.txt,München tazdpa – Die Mahnwache mahnte die Spru...,Munich tazdpa - The vigil warned the banners o...
4,1573221_full.txt,Berlin taz – Schon am kommenden Donnerstag wer...,"Berlin taz - The first refugees from Serbia, i..."


In [ ]:
# Häufigste Wörter insgesamt
def top_words(df_col, k):
    top_k = df_col.value_counts().head(k)
    print(f"\n{k} most frequent words:\n", top_k)
    return top_k

# TODO: Häufigste Wörter pro jahr 
def top_words_year(df, k):
    # TODO: find year, jahr aus id -> datei -> metadaten

    # group by year
    years = df.groupby(['year'])#.mean()
    # calc top words
    top_y = [(y[0],top_words(y[1],k)) for y in years]
    for y in top_y:
        print(f"\n{k} most frequent words:\n", y)
    return top_y

# TODO: Typ der Wörter (POS)

#### TFIDF of KI_sentences filtered with regex

In [ ]:
# compare KI_sentences and KI_sentences_full ids
ids_full = [x.replace("_full", "") for x in docs_full["id"].tolist()]
all_ids = docs_sents["id"].tolist() + ids_full
set_all_ids = set(all_ids)
non_repeated_ids = [x for x in set_all_ids if x not in ids_full] + [x for x in ids_full if x not in set_all_ids]
print(f"Number of ids: {len(all_ids)}, len unique ids: {len(set_all_ids)}, len non-repeated ids: {len(non_repeated_ids)}")
print("Non-repeated ids:", str(non_repeated_ids))
print("all ids:", str(set_all_ids))

# separate tfidf of ai-sentences and others


# combine

# calc tfidf on result

# visualize, ideally against other tfidf(s)


Number of ids: 1640, len unique: 578, len non-repeated: 428
Non-repeated ids: ['5722229.txt', '5822456.txt', '5184558.txt', '5454189.txt', '5455271.txt', '1902527.txt', '5906753.txt', '5829220.txt', '488760.txt', '5069570.txt', '5320885.txt', '5439902.txt', '1304884.txt', '5124622.txt', '5778640.txt', '5568940.txt', '0239343.txt', '5322497.txt', '5778108.txt', '1365278.txt', '489964.txt', '0408511.txt', '5452149.txt', '5050726.txt', '0308448.txt', '1185159.txt', '5321488.txt', '5896979.txt', '1607017.txt', '5193462.txt', '0316801.txt', '5167938.txt', '721946.txt', '5092968.txt', '5777808.txt', '0311380.txt', '5050516.txt', '5147336.txt', '5125642.txt', '5069001.txt', '481863.txt', '5458181.txt', '477818.txt', '5570975.txt', '592146.txt', '5192578.txt', '716793.txt', '5184586.txt', '5905402.txt', '480269.txt', '0314136.txt', '5022835.txt', '5453459.txt', '5723658.txt', '5320928.txt', '0308485.txt', '5778473.txt', '0319321.txt', '0241292.txt', '1185202.txt', '5783295.txt', '854933.txt', 

#### KI_sentence stats

In [9]:
# Häufigste Wörter pro satz / text

# Häufigkeit der einzelnen KI Terme

# average length of sentences / number of clauses

In [10]:
# number and type of verbs

# number and type of adjektives